In [12]:
from langchain_community.document_loaders import DirectoryLoader 
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
print('Loading the documents...')
def doc_loader(dir_path):
    documents = DirectoryLoader(path=dir_path,
                                loader_cls=PyMuPDFLoader,
                                show_progress=True).load()
    return documents
print('Documents loaded successfully.')

def split_docs(documents,chunk_size=1000,chunk_overlap=150):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(chunks)}")
    return chunks

def vector_db(chunks):
    print('Creating the vector database...')
    embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = Chroma.from_documents(chunks, embedding, persist_directory="./my_collection")
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    print(f'Total chunks in vector database: {len(vector_store)}')
    return retriever

def llm_model():
    print('Initializing the LLM model...')
    llm = ChatOpenAI(model="nvidia/nemotron-3-super-120b-a12b:free", 
                     api_key=os.getenv("OPENROUTER_API_KEY"),
                     base_url="https://openrouter.ai/api/v1")
    print('LLM model initialized successfully.')
    return llm

Loading the documents...
Documents loaded successfully.


### define state graph

In [14]:
from typing import TypedDict

class Graphstate(TypedDict):
    question:str
    documents:str
    generation:str

def retriever(state,retriever):
    question = state["question"]
    docs = retriever.invoke(question)

    return {
        "documents": docs,
        "question": question
    }

### document grade


In [15]:
# from pydantic import BaseModel

# class GradeDocument(BaseModel):
#     binary_score:str

# grade_prompt = ChatPromptTemplate.from_template(
# """
# you are a grader
# Document: {documents}
# question: {question}
# if document is relevant answer yes else no
# """
# )
# llm = llm_model()
# grade = llm.with_structured_output(GradeDocument)

### Query re-write (query reformulation)

In [16]:
geration_query = ChatPromptTemplate.from_template(
"""
rewite the user query to improve retrieval.
question:{question}
"""
)

def rewrite_query(state, llm):
    question= state["question"]
    response = llm.invoke(geration_query.format(question=question))
    print(f"Rewritten user query for better retrieval....")
    return {
        "question": response.content
        }


### Generation Final Answer

In [ ]:
final_prompt = ChatPromptTemplate.from_template(
"""
Answer using context only if you don't know the answer to the question just ask "you don't know"
context : {context}
question: {question}
"""
)

def gereration(state):
    question = state['question']
    docs = state['documents']
    context = "\n\n".join(doc.page_content for doc in docs)
    llm = llm_model()
    response = llm.invoke(final_prompt.format(context=context, question=question))
    return {
        "generation": response.content
        }

In [ ]:
def main():
    docs = doc_loader("../data")
    chunks = split_docs(docs)
    retrievered = vector_db(chunks)
    user_query = input('What do you wanna know about solo leveling? ')
    state = {
        "question": user_query
    }
    rewrite = rewrite_query(state, llm_model())
    answer = gereration(rewrite)
    print("Generating final answer...")
    print(f"Answer: {answer['generation']}")
if __name__ == "__main__":
    main()